# PCA + KNN on UCI Human Activity Recognition

This notebook mirrors the `main.py` pipeline for interactive exploration.
Run from the **project root** (`pca-har-KNN/`) so imports resolve, or add the root to `PYTHONPATH`.

In [7]:
import sys
from pathlib import Path

# Project root: parent of notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from sklearn.decomposition import PCA

from src.data_loader import load_har_splits
from src.evaluation import compute_metrics
from src.model import predict_time_knn, train_knn
from src.pca_module import apply_pca_2d, apply_pca_variance
from src.preprocessing import fit_transform_train_test
from src.visualization import plot_accuracy_comparison, plot_pca_2d_scatter

RNG = 42
label_universe = np.arange(1, 7)

## 1. Load data and standardize

In [8]:
X_train, X_test, y_train, y_test = load_har_splits()
print("Shapes:", X_train.shape, X_test.shape)

X_train_s, X_test_s, _ = fit_transform_train_test(X_train, X_test)

Shapes: (7352, 561) (2947, 561)


## 2. KNN without PCA

In [9]:
clf_base, t_fit = train_knn(X_train_s, y_train)
y_pred, t_pred = predict_time_knn(clf_base, X_test_s)
m_base = compute_metrics(y_test, y_pred, labels=label_universe)
print(f"Fit: {t_fit:.6f}s | Predict: {t_pred:.6f}s")
print("Accuracy:", m_base["accuracy"])

Fit: 0.041999s | Predict: 1.283985s
Accuracy: 0.8802171700033933


## 3. PCA (95% variance) + KNN

In [10]:
X_tr_pca, pca95, n_comp = apply_pca_variance(X_train_s, variance=0.95, random_state=RNG)
X_te_pca = pca95.transform(X_test_s)
print(f"561 → {n_comp} features (≈95% variance)")

clf_pca, t_fit_p = train_knn(X_tr_pca, y_train)
y_pred_p, t_pred_p = predict_time_knn(clf_pca, X_te_pca)
m_pca = compute_metrics(y_test, y_pred_p, labels=label_universe)
print(f"Fit: {t_fit_p:.6f}s | Predict: {t_pred_p:.6f}s")
print("Accuracy:", m_pca["accuracy"])

561 → 102 features (≈95% variance)
Fit: 0.006450s | Predict: 0.361953s
Accuracy: 0.8744485917882593
